# Genetic algorithm day-ahead scheduling

Generated notebook for the GridMind IS2 project. Run from the project root or use the first cell to configure the path.

In [ ]:
from pathlib import Path
import sys
from importlib import import_module
ROOT=Path.cwd()
if not (ROOT/'config.py').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
source=import_module('01_data_acquisition.data_source'); prep=import_module('02_data_preprocessing.preprocess'); ga=import_module('06_genetic_optimizer.ga_optimizer')

In [ ]:
raw=source.generate_building_data(days=10,seed=21)
clean=prep.clean_and_engineer(raw)
window=clean.tail(24)
load=window.load_kw.to_numpy(); solar=window.solar_kw.to_numpy(); price=window.price_per_kwh.to_numpy()

In [ ]:
result=ga.optimize_battery_schedule(load,solar,price,population_size=60,generations=40,seed=21)
print('Algorithm:',result['algorithm'])
print('Objective:',round(result['cost'],2))
print('Battery actions kW:',result['schedule'].round(2))

In [ ]:
import pandas as pd, plotly.express as px
schedule=pd.DataFrame({'timestamp':window.timestamp,'load_kw':load,'solar_kw':solar,'price_per_kwh':price,'battery_kw':result['schedule']})
px.bar(schedule,x='timestamp',y='battery_kw',title='Optimized battery schedule (positive=charge)').show()
schedule.head()

 Baseline energy-cost proxy and optimizer objective are different quantities due to penalties/terminal SOC.
baseline=float((load-solar).clip(min=0).dot(price))
{'baseline_energy_cost_proxy':baseline,'ga_full_objective':result['cost'],'throughput_kwh':result['detail']['throughput_kwh']}